In [4]:
import json
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient, models
from qdrant_client.models import SparseTextEmbedding
from openai import OpenAI
from deepeval import evaluate
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRecallMetric,
    ContextualPrecisionMetric
)
from deepeval.test_case import LLMTestCase
from deepeval.models.base_model import DeepEvalBaseLLM

In [5]:
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

In [6]:
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

Model for generting final answer

In [7]:
OPENROUTER_API_KEY = "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"

llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

FINAL_ANSWER_MODEL = "google/gemini-3-flash-preview"

In [8]:
def search(query: str, model: SentenceTransformer, collection: str, initial_k: int = 3):
    print(f"Запит: '{query}'")
    
    query_dense = model.encode(query).tolist()

    search_results = client.query_points(
        collection_name=collection,
        query=query_dense,
        limit=initial_k
    ).points

    if not search_results:
        return "На жаль, за цим запитом нічого не знайдено."

    documents = []
    for hit in search_results:
        text = hit.payload.get("text", "")
        source = hit.payload.get("file_name", "Невідомий документ")
        score = hit.score
        block = f"[Джерело: {source} | Score: {score:.2f}]\n{text}"
        documents.append(block)

    final_context = "\n\n---\n\n".join(documents)
    return final_context

In [9]:
def generate_final_answer(query: str, retrieved_context: str):
    system_prompt = """
    Ти є інтелектуальним помічником для працівників Українського Католицького Університету (УКУ).
    Твоє завдання — відповісти на запитання користувача, спираючись ВИКЛЮЧНО на наданий контекст з внутрішніх документів.

    ПРАВИЛА:
    1. Якщо в контексті немає відповіді, чесно скажи: "На жаль, у знайдених документах немає інформації для відповіді на це запитання." Не вигадуй жодних фактів.
    2. Формулюй відповідь чітко, професійно та структуровано.
    3. Обов'язково вказуй джерело інформації у форматі: [Джерело: Назва_файлу.pdf].
    """

    user_message = f"КОНТЕКСТ З БАЗИ ДАНИХ:\n{retrieved_context}\n\nЗАПИТАННЯ КОРИСТУВАЧА:\n{query}"

    response = llm_client.chat.completions.create(
        model=FINAL_ANSWER_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=0.1
    )
    return response.choices[0].message.content

In [11]:
with open("final_golden_dataset.json", "r", encoding="utf-8") as f:
    golden_data = json.load(f)

In [12]:
def rag_pipeline(query, model, collection):
    result = search(query, model, collection)
    answer = generate_final_answer(query, result)
    return result, answer

In [13]:
def create_test_cases(model, collection):
    test_cases = []

    for item in golden_data:
        
        contexts, prediction = rag_pipeline(item["input"], model, collection)
        if isinstance(contexts, str):
            contexts = [contexts]
        contexts = [str(c) for c in contexts]

        test_case = LLMTestCase(
            input=item["input"],
            actual_output=prediction,
            expected_output=item["expected_output"],
            retrieval_context=contexts
        )

        test_cases.append(test_case)

    return test_cases

In [14]:
class OpenRouterLLM(DeepEvalBaseLLM):

    def __init__(self):
        self.client = OpenAI(
            api_key=OPENROUTER_API_KEY,
            base_url="https://openrouter.ai/api/v1"
        )

    def load_model(self):
        return self.client

    def generate(self, prompt: str):
        response = self.client.chat.completions.create(
            model="openai/gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return response.choices[0].message.content

    async def a_generate(self, prompt: str):
        return self.generate(prompt)

    def get_model_name(self):
        return "openrouter-gpt"

In [15]:
judge_model = OpenRouterLLM()

In [16]:
metrics = [
    AnswerRelevancyMetric(model=judge_model),
    FaithfulnessMetric(model=judge_model),
    ContextualRecallMetric(model=judge_model),
    ContextualPrecisionMetric(model=judge_model)
]

<hr>

# paraphrase-multilingual-MiniLM-L12-v2

In [17]:
MODEL_BASELINE = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7615.71it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
COLLECTION_BASELINE = "ucu_documents_baseline"

In [19]:
evaluate(test_cases=create_test_cases(MODEL_BASELINE, COLLECTION_BASELINE), metrics=metrics)

Запит: 'На які категорії поділяються критерії відбору нових працівників в УКУ та яким документом вони регулюються?'
Запит: 'Яка процедура передбачена для чинного працівника УКУ, який бажає взяти участь у конкурсі на вакантну посаду всередині університету?'
Запит: 'Чому відзнаку «Викладач року» в УКУ не можна вважати звичайним професійним конкурсом і які обмеження встановлені щодо повторного отримання нагороди?'
Запит: 'Скільки осіб можуть стати лауреатами відзнаки «Викладач року» протягом одного року та чи можна отримати цю нагороду повторно?'
Запит: 'Опиши хронологію та ключові етапи процесу стратегічного планування цілей в УКУ.'
Запит: 'Яка процедура призначення керівника бакалаврської програми в УКУ згідно з положенням?'
Запит: 'Хто здійснює загальне керівництво магістерською підготовкою в УКУ та кому безпосередньо підпорядковується керівник магістерської програми?'
Запит: 'Хто очолює Бізнес школу УКУ згідно з управлінською структурою?'
Запит: 'За який термін працівника мають повідо

✨ You're running DeepEval's latest Answer Relevancy Metric! (using openrouter-gpt, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using openrouter-gpt, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using openrouter-gpt, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using openrouter-gpt, strict=False, 
async_mode=True)...

c:\Users\Olena\Downloads\ucu-employee-support-rag\.venv\lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Answer Relevancy (score: 0.9166666666666666, threshold: 0.5, strict: False, evaluation model: openrouter-gpt, reason: The score is 0.92 because the output provided relevant information about the selection criteria but included an irrelevant statement regarding differentiating employees by main place of work and part-time work, which does not directly address the question., error: None)
  - ✅ Faithfulness (score: 1.0, threshold: 0.5, strict: False, evaluation model: openrouter-gpt, reason: The score is 1.00 because there are no contradictions, indicating that the actual output aligns perfectly with the retrieval context. Great job maintaining consistency!, error: None)
  - ✅ Contextual Recall (score: 1.0, threshold: 0.5, strict: False, evaluation model: openrouter-gpt, reason: The score is 1.00 because the expected output clearly aligns with the relevant nodes in the retrieval context, specifically addressing selection criteria and hiring procedures, which are w

⚠ WARNING: No hyperparameters logged.
» ]8;id=217015;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 713.9s | token cost: None)
» Test Results (20 total tests):
   » Pass Rate: 60.0% | Passed: 12 | Failed: 8

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=0.9166666666666666, reason='The score is 0.92 because the output provided relevant information about the selection criteria but included an irrelevant statement regarding differentiating employees by main place of work and part-time work, which does not directly address the question.', strict_mode=False, evaluation_model='openrouter-gpt', error=None, evaluation_cost=None, verbose_logs='Statements:\n[\n    "Інформація щодо критеріїв відбору та регулювання процесу призначення працівників в УКУ є надана.",\n    "Питання відбору, призначення та вимог до працівників регулюються загальною Політикою з управління персоналом УКУ.",\n    "Політика з управління персоналом УКУ включає спеціалізовані положення.",\n    "Порядок обрання та прийняття на роботу науково-педагогічних працівників регулюється.",\n    "Існує положення про професійно

Evaluate generation while using another method for chunking

In [20]:
COLLECTION_BASELINE_RECURSIVE_CHAR = "ucu_documents_baseline_recursive_char"

In [21]:
evaluate(test_cases=create_test_cases(MODEL_BASELINE, COLLECTION_BASELINE_RECURSIVE_CHAR), metrics=metrics)

Запит: 'На які категорії поділяються критерії відбору нових працівників в УКУ та яким документом вони регулюються?'
Запит: 'Яка процедура передбачена для чинного працівника УКУ, який бажає взяти участь у конкурсі на вакантну посаду всередині університету?'
Запит: 'Чому відзнаку «Викладач року» в УКУ не можна вважати звичайним професійним конкурсом і які обмеження встановлені щодо повторного отримання нагороди?'
Запит: 'Скільки осіб можуть стати лауреатами відзнаки «Викладач року» протягом одного року та чи можна отримати цю нагороду повторно?'
Запит: 'Опиши хронологію та ключові етапи процесу стратегічного планування цілей в УКУ.'
Запит: 'Яка процедура призначення керівника бакалаврської програми в УКУ згідно з положенням?'
Запит: 'Хто здійснює загальне керівництво магістерською підготовкою в УКУ та кому безпосередньо підпорядковується керівник магістерської програми?'
Запит: 'Хто очолює Бізнес школу УКУ згідно з управлінською структурою?'
Запит: 'За який термін працівника мають повідо

✨ You're running DeepEval's latest Answer Relevancy Metric! (using openrouter-gpt, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using openrouter-gpt, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using openrouter-gpt, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using openrouter-gpt, strict=False, 
async_mode=True)...

c:\Users\Olena\Downloads\ucu-employee-support-rag\.venv\lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ❌ Answer Relevancy (score: 0.0, threshold: 0.5, strict: False, evaluation model: openrouter-gpt, reason: The score is 0.00 because the output contains multiple irrelevant statements that do not address the categories of selection criteria for new employees at УКУ, leading to a complete lack of relevance to the input question., error: None)
  - ❌ Faithfulness (score: 0.25, threshold: 0.5, strict: False, evaluation model: openrouter-gpt, reason: The score is 0.25 because the actual output fails to align with the retrieval context by omitting specific categories for selecting new employees, misrepresenting the focus on annual evaluations and accreditation, and neglecting to address disciplinary responsibilities mentioned in the context., error: None)
  - ✅ Contextual Recall (score: 1.0, threshold: 0.5, strict: False, evaluation model: openrouter-gpt, reason: The score is 1.00 because every aspect of the expected output is directly supported by the relevant nodes in 

⚠ WARNING: No hyperparameters logged.
» ]8;id=969437;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 728.03s | token cost: None)
» Test Results (20 total tests):
   » Pass Rate: 50.0% | Passed: 10 | Failed: 10

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=False, score=0.0, reason='The score is 0.00 because the output contains multiple irrelevant statements that do not address the categories of selection criteria for new employees at УКУ, leading to a complete lack of relevance to the input question.', strict_mode=False, evaluation_model='openrouter-gpt', error=None, evaluation_cost=None, verbose_logs='Statements:\n[\n    "У знайдених документах немає інформації щодо конкретних категорій відбору нових працівників.",\n    "Склад адміністрації УКУ та її повноваження щодо регулювання внутрішнього розпорядку.",\n    "Процес щорічної оцінки вже працюючих співробітників.",\n    "Атестація науково-педагогічних працівників.",\n    "Дисциплінарна відповідальність за порушення посадових інструкцій та цінностей Університету."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "no",\n        "reason": "Th